# Per-UID LSTM for IEEE-CIS Fraud — PyTorch version

This notebook mirrors `Time_Series_LSTM_per_UID.ipynb` (Keras) but is written in
PyTorch, following the structural pattern from the Kaggle reference:

> [arunmohan003 — *Sentiment analysis using LSTM - PyTorch*](https://www.kaggle.com/code/arunmohan003/sentiment-analysis-using-lstm-pytorch)

Same upstream logic as the Keras notebook:

- Load preprocessed checkpoints (`X_train_copy4.pkl`, `X_test_copy4.pkl`, `y_train.pkl`).
- Bridge train+test before windowing so test rows can use train history.
- Add three time-gap features per UID.
- Standardize (fit on train rows only).
- Build per-UID sliding windows of length `WINDOW`.
- Train under **strict expanding-window time validation** with `MIN_TRAIN_MONTHS = 3`.
- Save OOF and test predictions for ensembling with your XGBoost OOF.

## What's adapted from the sentiment-analysis reference

| arunmohan003's notebook | This notebook |
|--|--|
| Word indices `(batch, seq_len)` → `nn.Embedding` → `(batch, seq_len, embed_dim)` | Numeric features `(batch, seq_len, n_features)` directly into LSTM (no embedding) |
| `vocab_size`, `embedding_dim` hyperparameters | `n_features` only — no vocab |
| `nn.LSTM(input_size=embedding_dim, ...)` | `nn.LSTM(input_size=n_features, ...)` |
| `model.init_hidden(batch_size)` per iteration | Same pattern, kept for fidelity |
| `nn.BCELoss` after sigmoid | Same |
| Manual training loop with `optimizer.zero_grad()`, `loss.backward()`, `clip_grad_norm_`, `optimizer.step()` | Same |
| Best model saved by validation loss | Best model saved by **validation AUC** (better metric for fraud) |

The reason there's no embedding layer: in sentiment analysis each word is a discrete
token that has to be turned into a continuous vector. Your fraud features are already
continuous after `StandardScaler` (and previously-categorical fields like `card1` were
already integer-encoded by the upstream pipeline), so the LSTM can ingest them directly.


## MPS / Apple Silicon GPU notes

PyTorch supports Apple Silicon GPU via the **MPS** (Metal Performance Shaders) backend.
The config cell below picks the best available device automatically: `mps` if you're on
M-series, else `cuda`, else `cpu`.

A few specifics for MPS:

- Some ops fall back to CPU silently. For LSTM this works but you may see warnings
  about unsupported dtypes — mostly harmless.
- `torch.compile(...)` doesn't help much on MPS yet (Metal backend is limited),
  so this notebook doesn't use it.
- Mixed-precision (`autocast`) is supported but not used here — fp32 is more stable
  for LSTM on MPS, and the speed difference is small.
- `num_workers > 0` in `DataLoader` can deadlock on macOS in Jupyter. We use
  `num_workers=0` and rely on the unified-memory architecture for fast host-device
  transfer.


In [82]:
import sys, os
print(sys.executable)
print(os.environ.get("CONDA_DEFAULT_ENV"))

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/bin/python
Financial_Fraud_Detection_Thesis


In [83]:
# 0. Imports and config — PyTorch + MPS-aware
import os, gc, math, time, datetime, warnings, copy
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ----- Configuration -----
DATA_DIR = '/ieee-fraud-detection/pkl_exported_files'

WINDOW              = 20      # was 5  (change F)
MIN_TRAIN_MONTHS    = 3
BATCH               = 1024
EPOCHS              = 30      # was 12 (change I)
LR                  = 1e-3
WEIGHT_DECAY        = 2e-4
GRAD_CLIP           = 1.0     # was 0.5 (change D)
EARLY_STOP_PATIENCE = 6
SEED                = 42

HIDDEN_DIM   = 128
NUM_LAYERS   = 2
DROPOUT      = 0.3
N_SEEDS      = 3              # for seed ensembling (change H)
USE_POS_WEIGHT = False        # plain BCE for AUC (change C)
USE_STATIC_TOWER = True       # dual-tower (change B)

CACHE_DIR = "/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/sequence_cache_uid_disjoint/v6_current_transaction"

# ----- Reproducibility -----
torch.manual_seed(SEED); np.random.seed(SEED)

# ----- Device selection -----
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'PyTorch {torch.__version__}  device={device}')


PyTorch 2.11.0  device=mps


In [84]:
x = torch.randn(1024, 5, 250).to(device)
print(x.device)

mps:0


## 5. Build per-UID current-transaction windows

UPDATED v6: this variant keeps the UID-disjoint split from v4, but changes the
supervision target to **one scalar label per current transaction**.

Each sample is one transaction represented by that UID's history up to and
including the current transaction, right-padded to `MAX_LEN`. The model output is
one fraud logit per sample, not one logit per timestep. There are no padding
labels in v6 because each sequence has exactly one scalar target.


In [ ]:
## You would need to run the V6 version in Data_Processing_LSTM.inpynb first in order to extract the .npy files in folder v6_current_transaction to run the below code.

In [ ]:
from pathlib import Path
import numpy as np
CACHE_DIR = Path(CACHE_DIR)
def load(name, mmap=True):
    return np.load(CACHE_DIR / f"{name}.npy", mmap_mode="r" if mmap else None)

def load_orig_list(prefix):
    flat = load(f"{prefix}_flat", mmap=False)
    offsets = load(f"{prefix}_offsets", mmap=False)
    return [flat[offsets[i]:offsets[i + 1]] for i in range(len(offsets) - 1)]

# Big arrays
X_tr = load("X_tr").astype("float32")
X_va = load("X_va").astype("float32")

# Small arrays
L_tr = load("L_tr", mmap=False)
y_tr = load("y_tr", mmap=False)
uids_tr = load("uids_tr", mmap=False)

L_va = load("L_va", mmap=False)
y_va = load("y_va", mmap=False)
uids_va = load("uids_va", mmap=False)

# v4 orig lists
orig_tr = load_orig_list("orig_tr")
orig_va = load_orig_list("orig_va")

## 6. PyTorch `Dataset` and `DataLoader`

UPDATED v6: each dataset row is one current transaction window plus one scalar
fraud label. No timestep label mask is needed in this version.


In [ ]:
class CurrentTxnWindowDataset(Dataset):
    """One sample = one current transaction with UID history right-padded to MAX_LEN."""
    def __init__(self, X, L, y):
        self.X = torch.from_numpy(X)
        self.L = torch.from_numpy(L.astype('int64'))
        self.y = torch.from_numpy(y.astype('float32'))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.L[i], self.y[i]


def make_loader(X, L, y, batch_size, shuffle, device=None):
    return DataLoader(
        CurrentTxnWindowDataset(X, L, y),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )


## 7. PyTorch LSTM model

UPDATED v6: the model predicts only the **current transaction** for each UID
history window. It uses `pack_padded_sequence` on right-padded windows, then
combines the last real hidden state with mean+max pooling over the available
history. Because each sequence ends at the current transaction, this pooling does
not look into future transactions.


In [ ]:
class FraudLSTMCurrentTxn(nn.Module):
    """Scalar-output LSTM for the current transaction in a UID history window."""
    def __init__(self, n_features, hidden_dim=128, num_layers=2,
                 drop_prob=0.3, bidirectional=False):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=drop_prob if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        out_dim = hidden_dim * (2 if bidirectional else 1)

        # Last real hidden state + history mean pool + history max pool.
        head_in = out_dim * 3
        self.head = nn.Sequential(
            nn.Linear(head_in, 64), nn.ReLU(), nn.Dropout(drop_prob),
            nn.Linear(64, 1),
        )

    def forward(self, x, lengths):
        B, T, _ = x.shape
        lengths_dev = lengths.to(x.device)

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.detach().cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.lstm(packed)
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=T
        )

        idx = torch.arange(T, device=x.device).unsqueeze(0)
        mask = idx < lengths_dev.unsqueeze(1)
        mask_f = mask.unsqueeze(-1).float()

        row_idx = torch.arange(B, device=x.device)
        last_idx = (lengths_dev - 1).clamp(min=0)
        last_hidden = lstm_out[row_idx, last_idx, :]

        mean_pool = (lstm_out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)

        neg_inf = torch.finfo(lstm_out.dtype).min
        max_pool = lstm_out.masked_fill(~mask.unsqueeze(-1), neg_inf).max(dim=1).values

        feat = torch.cat([last_hidden, mean_pool, max_pool], dim=1)
        return self.head(feat).squeeze(-1)   # (B,)


In [ ]:
# smoke test
N_FEATURES = X_tr.shape[2]
m = FraudLSTMCurrentTxn(N_FEATURES).to(device)
xb = torch.randn(8, MAX_LEN, N_FEATURES, device=device)
lb = torch.randint(1, MAX_LEN+1, (8,), device='cpu')
print('forward output shape:', m(xb, lb).shape)   # expect (8,)
del m, xb, lb


## 8. Training loop (single UID-disjoint split, current-transaction prediction)

One model is trained on the ~80% train UIDs and evaluated on the held-out 20%.
Validation AUC is computed over one scalar prediction per validation transaction.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available()
                       else 'mps' if torch.backends.mps.is_available()
                       else 'cpu')
print('device:', device)

# Hyperparameters for the current-transaction setup
BATCH               = 64
EPOCHS              = 30
LR                  = 1e-3
WEIGHT_DECAY        = 2e-4
GRAD_CLIP           = 1.0
EARLY_STOP_PATIENCE = 6
HIDDEN_DIM          = 128
NUM_LAYERS          = 2
DROPOUT             = 0.3
BIDIRECTIONAL       = False
N_SEEDS             = 3


In [ ]:
def scalar_bce(logits, targets):
    return nn.functional.binary_cross_entropy_with_logits(logits, targets)


In [ ]:
def train_one_uid_run(X_tr, L_tr, y_tr,
                      X_va, L_va, y_va,
                      n_features, epochs, batch, lr, weight_decay,
                      device, patience, grad_clip, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FraudLSTMCurrentTxn(
        n_features=n_features,
        hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
        drop_prob=DROPOUT, bidirectional=BIDIRECTIONAL,
    ).to(device)

    train_loader = make_loader(X_tr, L_tr, y_tr, batch_size=batch, shuffle=True,  device=device)
    val_loader   = make_loader(X_va, L_va, y_va, batch_size=batch, shuffle=False, device=device)

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=epochs,
        steps_per_epoch=max(1, math.ceil(len(X_tr) / batch)),
        pct_start=0.1, anneal_strategy='cos',
    )

    best_auc, best_state, best_val_pred, bad = -1.0, None, None, 0
    for epoch in range(1, epochs + 1):
        model.train(); running, n_seen = 0.0, 0; t0 = time.time()
        for xb, lb, yb in train_loader:
            xb, lb = xb.to(device), lb.to(device)
            yb = yb.to(device)
            opt.zero_grad()
            logits = model(xb, lb)
            loss   = scalar_bce(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step(); sched.step()
            running += loss.item() * xb.size(0); n_seen += xb.size(0)
        train_loss = running / max(n_seen, 1)

        model.eval(); preds = []
        with torch.no_grad():
            for xb, lb, _ in val_loader:
                xb, lb = xb.to(device), lb.to(device)
                preds.append(torch.sigmoid(model(xb, lb)).cpu().numpy())
        val_pred = np.concatenate(preds)
        val_auc = roc_auc_score(y_va, val_pred)

        print(f'   ep {epoch:>2}/{epochs}  loss={train_loss:.4f}  '
              f'val_auc={val_auc:.4f}  ({time.time()-t0:.1f}s)')

        if val_auc > best_auc:
            best_auc      = val_auc
            best_state    = copy.deepcopy(model.state_dict())
            best_val_pred = val_pred
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f'   early stop at epoch {epoch}'); break

    if best_state is not None: model.load_state_dict(best_state)
    return best_val_pred, best_auc, model


In [ ]:
print(f'Current-transaction run with {N_SEEDS} seeds')
print(f'  train samples={X_tr.shape[0]:,}  val samples={X_va.shape[0]:,}')
print(f'  train UIDs={pd.Series(uids_tr).nunique():,}  val UIDs={pd.Series(uids_va).nunique():,}')

seed_val_preds, seed_aucs = [], []
SEED = 42
for s in range(N_SEEDS):
    seed = SEED + s
    print(f'\n-- seed {seed} --')
    val_p, val_auc, model = train_one_uid_run(
        X_tr, L_tr, y_tr,
        X_va, L_va, y_va,
        n_features=N_FEATURES,
        epochs=EPOCHS, batch=BATCH, lr=LR,
        weight_decay=WEIGHT_DECAY, device=device,
        patience=EARLY_STOP_PATIENCE, grad_clip=GRAD_CLIP, seed=seed,
    )
    seed_val_preds.append(val_p); seed_aucs.append(val_auc)
    del model
    if device.type == 'mps': torch.mps.empty_cache()
    gc.collect()

avg_val_pred = np.mean(seed_val_preds, axis=0)
overall_auc = roc_auc_score(y_va, avg_val_pred)
print(f'\n=== Per-seed AUCs: {[round(a,4) for a in seed_aucs]}')
print(f'=== Seed-averaged OOF AUC = {overall_auc:.4f} ===')


In [ ]:
# Map current-transaction predictions back to validation TransactionIDs
assert len(orig_va) == len(avg_val_pred), \
    f'len mismatch: {len(orig_va)} vs {len(avg_val_pred)}'

oof_df = pd.DataFrame({
    'TransactionID': orig_va,
    'oof_lstm_current_txn': avg_val_pred,
    'isFraud_label': y_va,
})
# oof_df.to_csv('oof_lstm_current_txn_uid_disjoint.csv', index=False)
# print(f'Saved {len(oof_df):,} OOF rows to oof_lstm_current_txn_uid_disjoint.csv')
print(f'  overall AUC: {roc_auc_score(oof_df.isFraud_label, oof_df.oof_lstm_current_txn):.4f}')


## 9. Save OOF predictions

v6 currently creates validation OOF predictions for the UID-disjoint split. Test
prediction is intentionally not wired here because this variant is for comparing
the scalar current-transaction objective against the v4 per-timestep objective.


In [ ]:
# Optional save:
# oof_df.to_csv('oof_lstm_current_txn_uid_disjoint.csv', index=False)
# print('Wrote oof_lstm_current_txn_uid_disjoint.csv')


## 10. Notes

**v6 summary**

- Starts from notebook v4 preprocessing and UID-disjoint validation.
- Changes the target shape from per-timestep `(batch, MAX_LEN)` to scalar `(batch,)`.
- Each sample is one current transaction plus that UID's available history up to the current row.
- Windows are right-padded and passed through `pack_padded_sequence`.
- Mean+max pooling is over the available history only; because the sequence ends at the current transaction, it does not include future transactions.
- There are no padding labels or label masks in v6. Each sequence has one `isFraud` target.

**Comparison target**

Use v6 to compare against v4 when you want to test whether scalar current-transaction prediction is better than per-timestep UID-sequence prediction under the same UID-disjoint validation idea.
